# 图的算法
图是一种应用广泛的数据结构，可以用来表示城市之间的道路、课程之间的先修关系、社交网络中的关注关系等。形式化地说，图由节点集合 $V$ 和边集合 $E$ 组成，记作 $(V, E)$。边可以看成节点集合上的二元关系 $(u, v) \in E \subseteq V \times V$。

如果边没有方向，$(u, v)$ 和 $(v, u)$ 表示同一条连接，这样的图称为无向图；如果边有方向，$(u, v)$ 表示从 `u` 指向 `v`，这样的图称为有向图。边还可以带权重，用来表示距离、费用、时间等代价。

下面先用邻接表实现一个 `Graph` 类。邻接表把每个节点映射到它的邻居集合，适合表示稀疏图；相比邻接矩阵，它通常能节省空间，并且可以较方便地遍历某个节点的所有出边。

In [6]:
class Graph:
    def __init__(self, directed=False):
        """
        初始化图
        :param directed: 是否为有向图，默认是无向图
        """
        self.directed = directed
        self.adj = {}  # 邻接表：{节点: {相邻节点: 权重}}

    def add_vertex(self, vertex):
        """
        添加节点
        """
        if vertex not in self.adj:
            self.adj[vertex] = {}

    def remove_vertex(self, vertex):
        """
        删除节点
        """
        if vertex not in self.adj:
            return

        # 删除该节点本身
        del self.adj[vertex]

        # 删除其他节点中指向该节点的边
        for v in self.adj:
            self.adj[v].pop(vertex, None)

    def add_edge(self, u, v, weight=1):
        """
        添加边
        :param u: 起点
        :param v: 终点
        :param weight: 边的权重，默认是 1
        """
        self.add_vertex(u)
        self.add_vertex(v)

        self.adj[u][v] = weight

        # 如果是无向图，需要反向也加一条边
        if not self.directed:
            self.adj[v][u] = weight

    def remove_edge(self, u, v):
        """
        删除边
        """
        if u in self.adj:
            self.adj[u].pop(v, None)

        if not self.directed and v in self.adj:
            self.adj[v].pop(u, None)

    def has_vertex(self, vertex):
        """
        判断节点是否存在
        """
        return vertex in self.adj

    def has_edge(self, u, v):
        """
        判断边是否存在
        """
        return u in self.adj and v in self.adj[u]

    def get_vertices(self):
        """
        获取所有节点
        """
        return list(self.adj.keys())

    def get_neighbors(self, vertex):
        """
        获取某个节点的所有邻居
        """
        if vertex not in self.adj:
            return []

        return list(self.adj[vertex].keys())

    def get_edge_weight(self, u, v):
        """
        获取边的权重
        """
        if self.has_edge(u, v):
            return self.adj[u][v]

        return None

    def get_edges(self):
        """
        获取所有边
        """
        edges = []

        for u in self.adj:
            for v, weight in self.adj[u].items():
                if self.directed:
                    edges.append((u, v, weight))
                else:
                    # 避免无向图重复输出边
                    if (v, u, weight) not in edges:
                        edges.append((u, v, weight))

        return edges

    def clear(self):
        """
        清空图
        """
        self.adj.clear()

    def __str__(self):
        """
        打印图
        """
        return str(self.adj)


## BFS与DFS
BFS（广度优先搜索）和 DFS（深度优先搜索）都是从一个起始节点出发，不断访问尚未访问过的邻居节点。二者都需要一个 `visited` 集合，避免在有环图中反复访问同一个节点。

它们的核心差异在于待访问节点集合的取出顺序。BFS 使用队列：先发现的节点先被扩展，因此遍历会一层一层向外推进；在无权图中，BFS 常用于求从起点到其他节点的最短边数。DFS 使用栈：后发现的节点先被扩展，因此会沿着一条路径尽量深入，走不下去后再回退；DFS 常用于连通性判断、环检测和生成遍历序列。

下面的 `bfs` 和 `dfs` 函数接收一个 `Graph` 对象和起始节点，并返回实际访问到的节点顺序。


In [7]:
from collections import deque


def bfs(graph, start):
    """
    广度优先遍历 BFS
    :param graph: Graph 对象
    :param start: 起始节点
    :return: 遍历顺序列表
    """
    if not graph.has_vertex(start):
        raise ValueError(f"The graph has no node {start}!")

    res = []
    frontiers = deque([start])
    visited = {start}

    while frontiers:
        vertex = frontiers.popleft()
        res.append(vertex)

        for neighbor in graph.get_neighbors(vertex):
            if neighbor in visited:
                continue

            visited.add(neighbor)
            frontiers.append(neighbor)

    return res


def dfs(graph, start):
    """
    深度优先遍历 DFS
    :param graph: Graph 对象
    :param start: 起始节点
    :return: 遍历顺序列表
    """
    if not graph.has_vertex(start):
        raise ValueError(f"The graph has no node {start}!")

    res = []
    frontiers = [start]
    visited = {start}

    while frontiers:
        vertex = frontiers.pop()
        res.append(vertex)

        for neighbor in reversed(graph.get_neighbors(vertex)):
            if neighbor in visited:
                continue

            visited.add(neighbor)
            frontiers.append(neighbor)

    return res


In [8]:
graph = Graph()
for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D")]:
    graph.add_edge(u, v)

print("BFS:", bfs(graph, "A"))
print("DFS:", dfs(graph, "A"))


BFS: ['A', 'B', 'C', 'D']
DFS: ['A', 'B', 'D', 'C']


## 拓扑排序
拓扑排序是针对有向无环图（DAG）的一种排序方式。它把图中的所有顶点排成一个线性序列，使得对于图中每一条有向边 $(u, v)$，顶点 `u` 一定排在 `v` 前面。换句话说，拓扑序描述的是一种满足所有依赖关系的执行顺序，例如课程先修关系、任务调度、编译依赖等。

如果图中存在有向环，就不存在合法的拓扑序。原因是环上的每个节点都依赖环中的另一个节点，无法找到一个真正可以排在最前面的节点。因此，拓扑排序不仅能给出顺序，也可以用来判断有向图中是否存在环。

Kahn 算法是一种常用的拓扑排序算法，核心思想是不断删除当前没有前置依赖的节点。具体流程如下：

1. 先统计每个顶点的入度。入度表示有多少条边指向该顶点，也就是它还有多少个前置依赖。
2. 把所有入度为 0 的顶点加入队列。它们没有未完成的前置依赖，可以最先进入拓扑序。
3. 每次从队列中取出一个顶点，将它加入结果序列；然后“删除”它发出的所有边，也就是把它所有后继节点的入度减 1。
4. 如果某个后继节点的入度因此变为 0，说明它的所有前置依赖都已经处理完，就把它加入队列。
5. 重复以上过程，直到队列为空。如果最终结果序列包含了图中的所有顶点，说明排序成功；否则说明图中仍有节点无法被删除，图中存在有向环。

下面的 `topological_sort` 函数接收一个有向 `Graph` 对象，并返回拓扑序列表。


In [9]:
from collections import deque


def topological_sort(graph):
    """
    使用 Kahn 算法进行拓扑排序
    :param graph: 有向 Graph 对象
    :return: 拓扑序列表
    """
    if not graph.directed:
        raise ValueError("Topological sort is only defined for directed graphs.")

    in_degree = {vertex: 0 for vertex in graph.adj}
    for u in graph.adj:
        for v in graph.adj[u]:
            in_degree[v] += 1

    frontiers = deque(vertex for vertex in graph.adj if in_degree[vertex] == 0)
    order = []

    while frontiers:
        vertex = frontiers.popleft()
        order.append(vertex)

        for neighbor in graph.get_neighbors(vertex):
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                frontiers.append(neighbor)

    if len(order) != len(graph.adj):
        raise ValueError("The graph contains a cycle; topological sort is not possible.")

    return order


In [10]:
dag = Graph(directed=True)
for u, v in [("A", "C"), ("B", "C"), ("B", "D"), ("C", "E"), ("D", "E")]:
    dag.add_edge(u, v)

print("Topological sort:", topological_sort(dag))


Topological sort: ['A', 'B', 'C', 'D', 'E']


## 连通分量与强连通分量
无向图里，如果任意两个顶点都互相可达，它们就在同一个连通分量里；这也常被叫作连通分支。图被分成若干个连通分量后，不同分量之间彼此不连通。

对有向图来说，常见概念有两类：

- 弱连通分量：把边的方向忽略后得到的连通分量。
- 强连通分量（SCC）：在有向图中，若任意两个顶点 `u` 和 `v` 都满足 `u` 可达 `v` 且 `v` 可达 `u`，它们就在同一个强连通分量里。

无向图的连通分量，以及有向图的弱连通分量，都可以直接用一次 BFS 或 DFS 求出。真正需要专门算法的是强连通分量：它把有向图压缩后会形成一张 DAG，因此常用于依赖分析、环检测、程序分析和网络分群。

求强连通分量的经典算法都能在 `O(V + E)` 时间内完成：

- Kosaraju 算法：两次 DFS。第一次记录完成时间，第二次在反图上按逆序搜索。实现直观，代码最容易讲清楚。
- Tarjan 算法：一次 DFS，维护 `dfn/lowlink` 和一个栈。它是最常见的单遍 SCC 算法之一。
- Gabow 算法：一次 DFS，维护两个栈 `S` 和 `P`。它不显式使用 `lowlink`，思路和 Tarjan 不同，但同样是线性时间。

下面给出三个算法的函数实现，返回值都是若干个强连通分量，每个分量是一个节点列表。

In [11]:
def kosaraju_scc(graph):
    """
    使用 Kosaraju 算法求强连通分量
    :param graph: 有向 Graph 对象
    :return: 若干个强连通分量，每个分量是一个节点列表
    """
    if not graph.directed:
        raise ValueError("Strongly connected components are only defined for directed graphs.")

    visited = set()
    order = []

    def dfs1(vertex):
        visited.add(vertex)
        for neighbor in graph.get_neighbors(vertex):
            if neighbor not in visited:
                dfs1(neighbor)
        order.append(vertex)

    for vertex in graph.get_vertices():
        if vertex not in visited:
            dfs1(vertex)

    reverse_adj = {vertex: [] for vertex in graph.get_vertices()}
    for u in graph.adj:
        for v in graph.adj[u]:
            reverse_adj[v].append(u)

    visited.clear()
    sccs = []

    def dfs2(vertex, component):
        visited.add(vertex)
        component.append(vertex)
        for neighbor in reverse_adj.get(vertex, []):
            if neighbor not in visited:
                dfs2(neighbor, component)

    for vertex in reversed(order):
        if vertex not in visited:
            component = []
            dfs2(vertex, component)
            sccs.append(component)

    return sccs


def tarjan_scc(graph):
    """
    使用 Tarjan 算法求强连通分量
    :param graph: 有向 Graph 对象
    :return: 若干个强连通分量，每个分量是一个节点列表
    """
    if not graph.directed:
        raise ValueError("Strongly connected components are only defined for directed graphs.")

    index = 0
    indices = {}
    lowlink = {}
    stack = []
    on_stack = set()
    sccs = []

    def dfs(vertex):
        nonlocal index
        indices[vertex] = index
        lowlink[vertex] = index
        index += 1
        stack.append(vertex)
        on_stack.add(vertex)

        for neighbor in graph.get_neighbors(vertex):
            if neighbor not in indices:
                dfs(neighbor)
                lowlink[vertex] = min(lowlink[vertex], lowlink[neighbor])
            elif neighbor in on_stack:
                lowlink[vertex] = min(lowlink[vertex], indices[neighbor])

        if lowlink[vertex] == indices[vertex]:
            component = []
            while True:
                node = stack.pop()
                on_stack.remove(node)
                component.append(node)
                if node == vertex:
                    break
            sccs.append(component)

    for vertex in graph.get_vertices():
        if vertex not in indices:
            dfs(vertex)

    return sccs


def gabow_scc(graph):
    """
    使用 Gabow 算法求强连通分量
    :param graph: 有向 Graph 对象
    :return: 若干个强连通分量，每个分量是一个节点列表
    """
    if not graph.directed:
        raise ValueError("Strongly connected components are only defined for directed graphs.")

    preorder = {}
    assigned = set()
    stack_s = []
    stack_p = []
    order = 0
    sccs = []

    def dfs(vertex):
        nonlocal order
        preorder[vertex] = order
        order += 1
        stack_s.append(vertex)
        stack_p.append(vertex)

        for neighbor in graph.get_neighbors(vertex):
            if neighbor not in preorder:
                dfs(neighbor)
            elif neighbor not in assigned:
                while preorder[stack_p[-1]] > preorder[neighbor]:
                    stack_p.pop()

        if stack_p and stack_p[-1] == vertex:
            stack_p.pop()
            component = []
            while True:
                node = stack_s.pop()
                assigned.add(node)
                component.append(node)
                if node == vertex:
                    break
            sccs.append(component)

    for vertex in graph.get_vertices():
        if vertex not in preorder:
            dfs(vertex)

    return sccs


## 最短路径
图的最短路径问题关心的是：从一个起点到一个终点，如何找到总代价最小的路径。这里的代价通常是边权之和，也可以表示距离、时间、费用等。

最短路径问题常分为两类：

- 单源最短路径：给定一个起点，求它到所有顶点的最短距离。
- 多源最短路径：求任意两点之间的最短距离。

### Dijkstra 算法
Dijkstra 算法是经典的单源最短路径算法，适用于**边权非负**的图。它的核心思想是贪心：每次从尚未确定最短路的顶点中，取出当前距离最小的顶点，并用它去松弛相邻顶点。由于边权非负，一旦某个顶点被取出，它的最短距离就不会再被更新。

适用条件：

- 可以处理有向图或无向图。
- 边权必须非负。

复杂度分析：

- 使用二叉堆时，时间复杂度为 `O((V + E) log V)`。
- 空间复杂度为 `O(V + E)`。

### Bellman-Ford 算法
Bellman-Ford 算法也是单源最短路径算法，但它允许存在**负权边**。它的核心思想是反复对所有边做松弛操作，总共重复 `V - 1` 轮。因为任意最短路径最多只会经过 `V - 1` 条边，所以做完这 `V - 1` 轮后，最短距离通常就已经稳定。

它还可以用于检测**负权环**：如果再额外做一轮松弛，距离仍然还能变小，就说明图中存在从起点可达的负权环，最短路不再有定义。

适用条件：

- 可以处理有向图或无向图。
- 可以包含负权边。
- 不能存在从源点可达的负权环。

复杂度分析：

- 时间复杂度为 `O(VE)`。
- 空间复杂度为 `O(V)`，如果只保留距离和前驱数组的话。

### Floyd 算法
Floyd 算法是典型的多源最短路径算法，也就是一次性求出任意两点之间的最短距离。它使用动态规划思想，依次枚举中转点 `k`，判断路径 `i -> k -> j` 是否能比当前的 `i -> j` 更短。

适用条件：

- 可以处理有向图或无向图。
- 可以存在负权边，但不能存在负权环。

复杂度分析：

- 时间复杂度为 `O(V^3)`。
- 空间复杂度为 `O(V^2)`。

一般来说，Dijkstra 更适合单源查询和稀疏图，Floyd 更适合顶点数不大但需要频繁查询任意两点最短路的场景。

In [12]:
from heapq import heappop, heappush


def dijkstra(graph, start):
    """
    使用 Dijkstra 算法求单源最短路径
    :param graph: Graph 对象
    :param start: 起始节点
    :return: dist, prev
    """
    if not graph.has_vertex(start):
        raise ValueError(f"The graph has no node {start}!")

    dist = {v: float('inf') for v in graph.get_vertices()}
    prev = {v: None for v in graph.get_vertices()}
    dist[start] = 0
    pq = [(0, start)]

    while pq:
        cur_dist, u = heappop(pq)
        if cur_dist != dist[u]:
            continue

        for v in graph.get_neighbors(u):
            weight = graph.get_edge_weight(u, v)
            if weight is None:
                continue
            if weight < 0:
                raise ValueError("Dijkstra algorithm requires non-negative edge weights.")

            nd = cur_dist + weight
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heappush(pq, (nd, v))

    return dist, prev


def bellman_ford(graph, start):
    """
    使用 Bellman-Ford 算法求单源最短路径
    :param graph: Graph 对象
    :param start: 起始节点
    :return: dist, prev
    """
    if not graph.has_vertex(start):
        raise ValueError(f"The graph has no node {start}!")

    vertices = graph.get_vertices()
    dist = {v: float('inf') for v in vertices}
    prev = {v: None for v in vertices}
    dist[start] = 0

    edges = graph.get_edges()
    if not graph.directed:
        expanded = []
        for u, v, w in edges:
            expanded.append((u, v, w))
            expanded.append((v, u, w))
        edges = expanded

    for _ in range(len(vertices) - 1):
        updated = False
        for u, v, weight in edges:
            if dist[u] == float('inf'):
                continue
            nd = dist[u] + weight
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                updated = True
        if not updated:
            break

    for u, v, weight in edges:
        if dist[u] != float('inf') and dist[u] + weight < dist[v]:
            raise ValueError("The graph contains a negative cycle reachable from the source.")

    return dist, prev


def floyd(graph):
    """
    使用 Floyd 算法求任意两点最短路径
    :param graph: Graph 对象
    :return: dist, nxt
    """
    vertices = graph.get_vertices()
    dist = {u: {v: float('inf') for v in vertices} for u in vertices}
    nxt = {u: {v: None for v in vertices} for u in vertices}

    for v in vertices:
        dist[v][v] = 0
        nxt[v][v] = v

    for u in graph.adj:
        for v, weight in graph.adj[u].items():
            if weight < dist[u][v]:
                dist[u][v] = weight
                nxt[u][v] = v

    for k in vertices:
        for i in vertices:
            if dist[i][k] == float('inf'):
                continue
            for j in vertices:
                nd = dist[i][k] + dist[k][j]
                if nd < dist[i][j]:
                    dist[i][j] = nd
                    nxt[i][j] = nxt[i][k]

    for v in vertices:
        if dist[v][v] < 0:
            raise ValueError("The graph contains a negative cycle; Floyd shortest paths are undefined.")

    return dist, nxt


def reconstruct_path(nxt, start, end):
    """
    根据 Floyd 的 next 矩阵重建路径
    """
    if nxt[start][end] is None:
        return []

    path = [start]
    while start != end:
        start = nxt[start][end]
        if start is None:
            return []
        path.append(start)

    return path


## 最小生成树
最小生成树（Minimum Spanning Tree, MST）是无向连通带权图中的一个子图。它需要连接图中的所有顶点，并且边权总和最小。因为它是一棵树，所以如果图中有 `V` 个顶点，那么最小生成树一定包含 `V - 1` 条边。

最小生成树问题只针对**无向图**讨论。如果原图不连通，就不存在覆盖所有顶点的一棵生成树，只能得到每个连通分量内部的最小生成森林。

最小生成树有几个常用性质：

- 无环性：最小生成树中不会出现环，否则可以删掉环上一条边，仍然保持连通。
- 连通性：最小生成树必须连接所有顶点。
- 边数性质：包含 `V` 个顶点的生成树一定有 `V - 1` 条边。
- 切分性质：把顶点集合切成两部分，跨越这个切分的最小权重边一定可以属于某棵最小生成树。
- 环性质：任意一个环中，权重最大的边一定不必出现在某棵最小生成树中。

### Prim 算法
Prim 算法从一个起点开始，维护一个已经加入生成树的顶点集合。每一步选择一条连接“树内顶点”和“树外顶点”的最小权重边，把新的顶点加入生成树。它的思路更像是从一个局部连通块不断向外扩张。

复杂度分析：

- 使用二叉堆和邻接表时，时间复杂度为 `O(E log E)`，也常写作 `O(E log V)`。
- 空间复杂度为 `O(V + E)`。

### Kruskal 算法
Kruskal 算法先把所有边按权重从小到大排序，然后依次尝试加入这些边。只要加入某条边不会形成环，就把它放进生成树。为了高效判断是否成环，通常使用并查集维护顶点所属的连通分量。

复杂度分析：

- 排序所有边需要 `O(E log E)` 时间，并查集操作几乎是常数时间，因此总时间复杂度为 `O(E log E)`。
- 空间复杂度为 `O(V + E)`。

Prim 更适合从某个顶点开始逐步扩展；Kruskal 更适合从全局边集出发，按边权从小到大构造生成树。

In [13]:
from heapq import heappop, heappush


def prim_mst(graph, start=None):
    """
    使用 Prim 算法求最小生成树
    :param graph: 无向 Graph 对象
    :param start: 起始节点，默认使用第一个节点
    :return: mst_edges, total_weight
    """
    if graph.directed:
        raise ValueError("Minimum spanning tree is only defined for undirected graphs.")

    vertices = graph.get_vertices()
    if not vertices:
        return [], 0

    if start is None:
        start = vertices[0]
    if not graph.has_vertex(start):
        raise ValueError(f"The graph has no node {start}!")

    visited = {start}
    pq = []
    mst_edges = []
    total_weight = 0

    for neighbor in graph.get_neighbors(start):
        weight = graph.get_edge_weight(start, neighbor)
        heappush(pq, (weight, start, neighbor))

    while pq and len(visited) < len(vertices):
        weight, u, v = heappop(pq)
        if v in visited:
            continue

        visited.add(v)
        mst_edges.append((u, v, weight))
        total_weight += weight

        for neighbor in graph.get_neighbors(v):
            if neighbor not in visited:
                edge_weight = graph.get_edge_weight(v, neighbor)
                heappush(pq, (edge_weight, v, neighbor))

    if len(visited) != len(vertices):
        raise ValueError("The graph is not connected; a spanning tree does not exist.")

    return mst_edges, total_weight


def kruskal_mst(graph):
    """
    使用 Kruskal 算法求最小生成树
    :param graph: 无向 Graph 对象
    :return: mst_edges, total_weight
    """
    if graph.directed:
        raise ValueError("Minimum spanning tree is only defined for undirected graphs.")

    parent = {}
    rank = {}

    def make_set(x):
        parent[x] = x
        rank[x] = 0

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        root_x = find(x)
        root_y = find(y)
        if root_x == root_y:
            return False

        if rank[root_x] < rank[root_y]:
            parent[root_x] = root_y
        elif rank[root_x] > rank[root_y]:
            parent[root_y] = root_x
        else:
            parent[root_y] = root_x
            rank[root_x] += 1

        return True

    vertices = graph.get_vertices()
    for vertex in vertices:
        make_set(vertex)

    mst_edges = []
    total_weight = 0
    edges = sorted(graph.get_edges(), key=lambda edge: edge[2])

    for u, v, weight in edges:
        if union(u, v):
            mst_edges.append((u, v, weight))
            total_weight += weight
            if len(mst_edges) == len(vertices) - 1:
                break

    if len(mst_edges) != len(vertices) - 1:
        raise ValueError("The graph is not connected; a spanning tree does not exist.")

    return mst_edges, total_weight
